In [161]:
import duckdb
import pandas as pd

In [ ]:
con = duckdb.connect("/home/etienne/projects/inatML/data/raw/raw.duckdb")
df = con.execute("SELECT * FROM downloads").df()
con.close()

In [163]:
print_obs = lambda df: print( "Total observations:", df['obs_count'].sum())


In [164]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175044 entries, 0 to 175043
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   id                             175044 non-null  int64         
 1   uuid                           175044 non-null  str           
 2   observed_on_string             174622 non-null  str           
 3   observed_on                    174622 non-null  datetime64[us]
 4   time_observed_at               172818 non-null  str           
 5   time_zone                      175037 non-null  str           
 6   user_id                        175044 non-null  int64         
 7   created_at                     175044 non-null  str           
 8   updated_at                     175044 non-null  str           
 9   quality_grade                  175044 non-null  str           
 10  license                        139584 non-null  str           
 11  url        

In [165]:
#Observation count filter

threshold = 100

unique_users =  df['user_id'].nunique()
df_user = df.groupby(by ='user_id').count().reset_index('user_id')
df_user = df_user[['user_id', 'id']].rename(columns={'id': 'obs_count'})
df_user.sort_values(by='obs_count', inplace=True, ascending= False)
df_user = df_user[df_user['obs_count'] >= threshold]

print("Unique users:", unique_users)
print("Unique users with over 20:", df_user.shape[0])
print_obs(df_user)

Unique users: 8983
Unique users with over 20: 244
Total observations: 110853


In [166]:
#Temporal filter
old_threshold = '2018-01-01'
new_threshold = '2024-01-01'
df_temporal = pd.merge(df, df_user, on=['user_id'], how='inner')
df_temporal

min = df_temporal[['user_id', 'observed_on']].groupby(by='user_id').min().reset_index('user_id').rename(columns={'observed_on' : 'oldest'})
max = df_temporal[['user_id', 'observed_on']].groupby(by='user_id').max().reset_index('user_id').rename(columns={'observed_on' : 'newest'})
df_user = pd. merge(df_user, min, on=['user_id'], how='inner')
df_user = pd. merge(df_user, max, on=['user_id'], how='inner')

df_user = df_user[df_user['oldest'] <= old_threshold]
print("User with obs older than 2018: ", df_user.shape[0])
print_obs(df_user)
df_user = df_user[df_user['newest'] >= new_threshold]
print("User with last obs at least in last 2 years: ", df_user.shape[0])
print_obs(df_user)

df_user


User with obs older than 2018:  53
Total observations: 48062
User with last obs at least in last 2 years:  49
Total observations: 47559


,user_id,obs_count,oldest,newest
0,170921,12032,2007-05-08,2026-02-13
1,2129201,7438,2016-06-22,2025-11-02
3,696204,4087,1969-08-15,2025-10-31
7,2463161,2743,1968-06-23,2025-11-06
8,164015,2205,2011-06-05,2024-02-27
12,3267913,1476,2009-08-30,2025-11-16
13,1074380,1427,2004-06-20,2026-01-08
14,4379636,1286,2010-07-01,2025-09-14
18,776355,886,2017-05-12,2025-10-06
19,159430,835,1990-07-07,2025-09-29
